In [6]:
import sys
from IPython.display import display, Javascript

def restart_kernel():
    """Restart the Jupyter Notebook kernel to reflect changes in modules and packages."""
    display(Javascript("Jupyter.notebook.kernel.restart()"))
    print("Kernel is restarting...")

restart_kernel()

import numpy as np
import torch
import tensorly as tl
from tensorly.decomposition import parafac
import sparse
import pickle
import os
import numpy.random as rn
import scipy.stats as st
from tensorly.cp_tensor import CPTensor

from bptf import BPTF as BPTF
import bptf

# Please run everything up to the first bptf fit
# Thereafter, run each following cell on their own

<IPython.core.display.Javascript object>

Kernel is restarting...


# Helper functions

In [7]:
def generate(shp=(30, 30, 20, 10), K=5, alpha=0.1, beta=0.1):
    """Generate a count tensor from the BPTF model.

    PARAMS:
    shp -- (tuple) shape of the generated count tensor
    K -- (int) number of latent components
    alpha -- (float) shape parameter of gamma prior over factors
    beta -- (float) rate parameter of gamma prior over factors

    RETURNS:
    Mu -- (np.ndarray) true Poisson rates
    Y -- (np.ndarray) generated count tensor
    """
    Theta_DK_M = [rn.gamma(alpha, 1./beta, size=(D, K)) for D in shp]
    Mu = tl.cp_to_tensor(CPTensor((None, Theta_DK_M)))
    assert Mu.shape == shp
    Y = rn.poisson(Mu)
    return Mu, Y

# Load data

In [8]:
use_existing_data = False

# for the first bptf.fit:
# using seed 100 causes the assert delta >= 0 to trip
# but using seed 0 doesn't
# please swap between the 2 seeds to get the 2 different types of assertion errors in the other cells
rn.seed(100)
num_months = 12

if use_existing_data:
    assert os.path.exists('sptensor.pkl'), 'No such file.'
    with open('sptensor.pkl', 'rb') as f:
        data = pickle.load(f)
    data = data[:, :, :, :num_months, :]
    data = sparse.COO(data)
else:
    data = generate(shp=(200, 200, 20, num_months, 3), K=10)[1]
    data = sparse.COO(data)

n_components = 10
max_iter = 500
tol = 1e-6

# Building mask (base example)

In [9]:
mask = np.zeros(data.shape)
# april of GDELT is set to missing
mask[:, :, :, 3, 1] = 1

# diagonals are set to missing
mask[np.eye(mask.shape[0]).astype(bool)] = 1
mask = sparse.COO(mask.astype(np.int64))

# Fit model

In [10]:
BPTF_model = BPTF(data_shape=data.shape, n_components=n_components)
BPTF_model.fit(data, mask = mask, max_iter = max_iter, verbose=False, missing_val=1, tol=tol)

  0%|          | 0/500 [00:00<?, ?it/s]

alpha * beta = 0.099144977795355
Number of nonpositive shape parameters = 0
Smallest shape element = 28.23028580973734
Number of nonpositive rate parameters = 0
Smallest rate element = 128280.61536618976
alpha * beta = 0.09869065968855914
Number of nonpositive shape parameters = 0
Smallest shape element = 28.13266211919287
Number of nonpositive rate parameters = 0
Smallest rate element = 127805.94897155576
alpha * beta = 0.10072429442248375
Number of nonpositive shape parameters = 0
Smallest shape element = 6545.648060361064
Number of nonpositive rate parameters = 0
Smallest rate element = 1207841.4277987513
alpha * beta = 0.09778941418569012
Number of nonpositive shape parameters = 0
Smallest shape element = 21372.694599051858
Number of nonpositive rate parameters = 0
Smallest rate element = 1219326.9966603261
alpha * beta = 0.09918556796501184
Number of nonpositive shape parameters = 0
Smallest shape element = 202836.4895528308
Number of nonpositive rate parameters = 0
Smallest rate 

  0%|          | 1/500 [00:10<1:23:16, 10.01s/it]

alpha * beta = 0.0869071135500356
Number of nonpositive shape parameters = 0
Smallest shape element = 11.994756813809177
Number of nonpositive rate parameters = 0
Smallest rate element = 79494.12540648413
alpha * beta = 0.09943567799479071
Number of nonpositive shape parameters = 0
Smallest shape element = 22.376695365802146
Number of nonpositive rate parameters = 0
Smallest rate element = 83695.2464773149
alpha * beta = 0.10213067886538935
Number of nonpositive shape parameters = 0
Smallest shape element = 1818.749182367177
Number of nonpositive rate parameters = 0
Smallest rate element = 678851.3654480305
alpha * beta = 0.10086740166791944
Number of nonpositive shape parameters = 0
Smallest shape element = 985.7282431939994
Number of nonpositive rate parameters = 0
Smallest rate element = 391441.11450367264
alpha * beta = 0.10140615601501145
Number of nonpositive shape parameters = 0
Smallest shape element = 11158.492153823066
Number of nonpositive rate parameters = 0
Smallest rate e

  0%|          | 2/500 [00:20<1:23:19, 10.04s/it]

...

  6%|▌         | 29/500 [05:19<1:26:20, 11.00s/it]

alpha * beta = 0.10600013891548982
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000000082943837
Number of nonpositive rate parameters = 0
Smallest rate element = 6722.5904313863475
alpha * beta = 0.11693213065697701
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000000019487354
Number of nonpositive rate parameters = 0
Smallest rate element = 4948.9976836535325
alpha * beta = 0.11827020599711807
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000113029731153
Number of nonpositive rate parameters = 0
Smallest rate element = 60970.7068540834


C:\Users\luiyu\Documents\aaron_schein\bptf_new\bptf\src\bptf\bptf.py:241: RuntimeWarning: invalid value encountered in log
  self.G_DK_M[m] = np.exp(sp.psi(shp_DK) - np.log(rte_DK))
  6%|▌         | 29/500 [05:26<1:28:29, 11.27s/it]

alpha * beta = 9.180360258887749e-14
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000121629510053
Number of nonpositive rate parameters = 1
Smallest rate element = -1.8625533456283682e-09


AssertionError: 

# Mask with last mode completely masked

In [11]:
mask = np.zeros(data.shape)
# april of GDELT is set to missing
mask[:, :, :, 3, :] = 1

# diagonals are set to missing
mask[np.eye(mask.shape[0]).astype(bool)] = 1
mask = sparse.COO(mask.astype(np.int64))

mask = (1 - mask.todense()).astype(np.int64)
mask = sparse.COO(mask)

BPTF_model = BPTF(data_shape=data.shape, n_components=n_components)
BPTF_model.fit(data, mask = mask, max_iter = max_iter, verbose=False, missing_val=0, tol=tol)

  0%|          | 0/500 [00:00<?, ?it/s]

alpha * beta = 0.09893741647391313
Number of nonpositive shape parameters = 0
Smallest shape element = 32.27410721592383
Number of nonpositive rate parameters = 0
Smallest rate element = 120416.40332355326
alpha * beta = 0.09854154797801312
Number of nonpositive shape parameters = 0
Smallest shape element = 29.283204924305295
Number of nonpositive rate parameters = 0
Smallest rate element = 133416.94884265083
alpha * beta = 0.09925463471488502
Number of nonpositive shape parameters = 0
Smallest shape element = 7202.781450829711
Number of nonpositive rate parameters = 0
Smallest rate element = 1186908.4543073273
alpha * beta = 0.10063208204295267
Number of nonpositive shape parameters = 0
Smallest shape element = 22472.581534336045
Number of nonpositive rate parameters = 0
Smallest rate element = 0.10063208204295267
alpha * beta = 0.09820754707076512
Number of nonpositive shape parameters = 0
Smallest shape element = 232179.05914437602
Number of nonpositive rate parameters = 0
Smallest 

  0%|          | 1/500 [01:41<14:00:25, 101.05s/it]

alpha * beta = 0.08204882643517117
Number of nonpositive shape parameters = 0
Smallest shape element = 13.567816814154
Number of nonpositive rate parameters = 0
Smallest rate element = 85078.88055583787
alpha * beta = 0.09965031396422362
Number of nonpositive shape parameters = 0
Smallest shape element = 23.698339275687207
Number of nonpositive rate parameters = 0
Smallest rate element = 100200.99494699496
alpha * beta = 0.10097334700966079
Number of nonpositive shape parameters = 0
Smallest shape element = 1403.5998047578353
Number of nonpositive rate parameters = 0
Smallest rate element = 997040.535458147
alpha * beta = 1.567915759555907e-07
Number of nonpositive shape parameters = 0
Smallest shape element = 950.2795216969344
Number of nonpositive rate parameters = 0
Smallest rate element = 1.567915759555907e-07
alpha * beta = 0.09676038207271676
Number of nonpositive shape parameters = 0
Smallest shape element = 11538.551322664853
Number of nonpositive rate parameters = 0
Smallest r

  0%|          | 2/500 [03:24<14:10:11, 102.43s/it]

...

alpha * beta = 0.09903223316486019
Number of nonpositive shape parameters = 0
Smallest shape element = 0.1000000000832283
Number of nonpositive rate parameters = 0
Smallest rate element = 1263.8233324454777
alpha * beta = 0.12549944704250535
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000000000204548
Number of nonpositive rate parameters = 0
Smallest rate element = 1253.2972108997
alpha * beta = 0.12098061057037449
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000064049434217
Number of nonpositive rate parameters = 0
Smallest rate element = 13808.368715455483
alpha * beta = 7.921316916367615e-240
Number of nonpositive shape parameters = 0
Smallest shape element = 0.1
Number of nonpositive rate parameters = 0
Smallest rate element = 7.921316916367615e-240
alpha * beta = 1.5118437210679592e-19
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000484963467433
Number of nonpositive rate parameters = 0
Smallest rate

  8%|▊         | 42/500 [1:14:32<13:27:47, 105.82s/it]

alpha * beta = 0.0988571631384686
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000000008322599
Number of nonpositive rate parameters = 0
Smallest rate element = 1264.9469881378934
alpha * beta = 0.12550112290397505
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000000000204552
Number of nonpositive rate parameters = 0
Smallest rate element = 1253.3439680091928
alpha * beta = 0.12098239853246105
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000064055180413
Number of nonpositive rate parameters = 0
Smallest rate element = 13809.56388933437


  8%|▊         | 42/500 [1:15:35<13:44:18, 107.99s/it]


KeyboardInterrupt: 

# Mask with the 1st and 3rd indices masked

In [ ]:
mask = np.zeros(data.shape)
# april is set to missing
mask[:, :, :, 3, [0, 2]] = 1

# diagonals are set to missing
mask[np.eye(mask.shape[0]).astype(bool)] = 1
mask = sparse.COO(mask.astype(np.int64))

BPTF_model = BPTF(data_shape=data.shape, n_components=n_components)
BPTF_model.fit(data, mask = mask, max_iter = max_iter, verbose=False, missing_val=1, tol=tol)

  0%|          | 0/500 [00:00<?, ?it/s]

alpha * beta = 0.09865682141427463
Number of nonpositive shape parameters = 0
Smallest shape element = 7.417720486353945
Number of nonpositive rate parameters = 0
Smallest rate element = 238635.3581338652
alpha * beta = 0.09884390950492251
Number of nonpositive shape parameters = 0
Smallest shape element = 29.320886858761703
Number of nonpositive rate parameters = 0
Smallest rate element = 18219.463467163052
alpha * beta = 0.09914875541517927
Number of nonpositive shape parameters = 0
Smallest shape element = 3530.076042279061
Number of nonpositive rate parameters = 0
Smallest rate element = 195600.2461161263
alpha * beta = 0.09931330031906625
Number of nonpositive shape parameters = 0
Smallest shape element = 71.19265411108493
Number of nonpositive rate parameters = 0
Smallest rate element = 50816.82731742758
alpha * beta = 0.09728433860884539
Number of nonpositive shape parameters = 0
Smallest shape element = 262373.64075628493
Number of nonpositive rate parameters = 0
Smallest rate 

  0%|          | 1/500 [00:07<1:06:23,  7.98s/it]

alpha * beta = 1.3156603734846892
Number of nonpositive shape parameters = 0
Smallest shape element = 4.83724723451653
Number of nonpositive rate parameters = 0
Smallest rate element = 239960.22354439853
alpha * beta = 0.0989925862227331
Number of nonpositive shape parameters = 0
Smallest shape element = 10.389451055078247
Number of nonpositive rate parameters = 0
Smallest rate element = 16620.121668245287
alpha * beta = 0.09911154551668355
Number of nonpositive shape parameters = 0
Smallest shape element = 1071.2580798216013
Number of nonpositive rate parameters = 0
Smallest rate element = 144474.643146387
alpha * beta = 0.10057704165043309
Number of nonpositive shape parameters = 0
Smallest shape element = 17.47112625192181
Number of nonpositive rate parameters = 0
Smallest rate element = 31908.22222471071
alpha * beta = 0.09823499791389008
Number of nonpositive shape parameters = 0
Smallest shape element = 13541.992360319917
Number of nonpositive rate parameters = 0
Smallest rate el

  0%|          | 2/500 [00:16<1:07:21,  8.12s/it]

...

alpha * beta = 1.4669840498748714
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000000731934684
Number of nonpositive rate parameters = 0
Smallest rate element = 2787.831304880402
alpha * beta = 0.11056233783762204
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000000650439772
Number of nonpositive rate parameters = 0
Smallest rate element = 198.7679022814731
alpha * beta = 0.11294246052705144
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000281169053625
Number of nonpositive rate parameters = 0
Smallest rate element = 2440.81871351428
alpha * beta = 8.634016340370386e-11
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000004669281277
Number of nonpositive rate parameters = 0
Smallest rate element = 1.0858542010610695e-07
alpha * beta = 0.03949308347633759
Number of nonpositive shape parameters = 0
Smallest shape element = 0.6798632840116212
Number of nonpositive rate parameters = 0
Sma

  2%|▏         | 10/500 [01:21<1:06:33,  8.15s/it]

alpha * beta = 1.4704947461036424
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000000743414378
Number of nonpositive rate parameters = 0
Smallest rate element = 2310.1350975827654
alpha * beta = 0.11083027067567369
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000000650573912
Number of nonpositive rate parameters = 0
Smallest rate element = 164.57667014559155
alpha * beta = 0.11312240328548549
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000136015609182
Number of nonpositive rate parameters = 0
Smallest rate element = 2201.779353555959
alpha * beta = 7.104830565985153e-12
Number of nonpositive shape parameters = 0
Smallest shape element = 0.10000004500546161
Number of nonpositive rate parameters = 0
Smallest rate element = 2.612430870102077e-07
alpha * beta = 0.027019348428500085
Number of nonpositive shape parameters = 0
Smallest shape element = 0.15946039714778149
Number of nonpositive rate parameters = 0

  2%|▏         | 10/500 [01:29<1:12:46,  8.91s/it]


AssertionError: delta = -0.0004582024948243403

# Mask with only diagonals masked

In [ ]:
mask = np.zeros(data.shape)

# diagonals are set to missing
mask[np.eye(mask.shape[0]).astype(bool)] = 1
mask = sparse.COO(mask.astype(np.int64))

BPTF_model = BPTF(data_shape=data.shape, n_components=n_components)
BPTF_model.fit(data, mask = mask, max_iter = max_iter, verbose=False, missing_val=1, tol=tol)

100%|██████████| 500/500 [56:35<00:00,  6.79s/it]


BPTF(data_shape=(200, 200, 20, 24, 3), n_components=10)